In [0]:
from pyspark.sql.functions import current_timestamp, col, lit

# Read yellow taxi data
yellow_df = spark.read.parquet("/Volumes/workspace/taxi_schema/taxi_raw_data/")

# Add audit metadata and standardize column names
df_bronze = (yellow_df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("taxi_type", lit("yellow"))
    .withColumn("pickup_datetime", col("tpep_pickup_datetime"))
    .withColumn("dropoff_datetime", col("tpep_dropoff_datetime"))
    .drop("tpep_pickup_datetime", "tpep_dropoff_datetime")
)

# Write to bronze Delta table
(df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.taxi_schema.taxi_bronze")
)

print(f"Bronze table created with {df_bronze.count()} records")